# Vector Geospatial Data

## Overview

Vector data represents geographic features as points, lines, and polygons with associated attributes. GeoPandas extends pandas to handle spatial data natively, combining the tabular power of DataFrames with geometry operations.

**Core geometry types:**

| Type | Represents | Example |
|---|---|---|
| Point | Single location | Monitoring site |
| LineString | Linear feature | River reach, transect |
| Polygon | Area feature | Catchment, buffer zone |
| MultiPolygon | Multiple polygons | Fragmented habitat |

**Coordinate Reference Systems (CRS):**
- Geographic (WGS84, EPSG:4326): degrees lon/lat — for plotting, web maps
- Projected (e.g. BNG EPSG:27700, UTM): metres — for distance and area calculations
- Always reproject to a metric CRS before computing distances, areas, or buffers

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
from shapely.geometry import Point, LineString, Polygon, MultiPolygon
from shapely.ops import unary_union
import warnings; warnings.filterwarnings('ignore')

rng = np.random.default_rng(42)

# Simulate monitoring sites in a Scottish river catchment (approx BNG coords)
# British National Grid EPSG:27700: easting/northing in metres
sites_data = {
    'site_id':   ['NORTH_01','NORTH_02','NORTH_03','SOUTH_01','SOUTH_02',
                  'MAIN_01','MAIN_02','OUTLET'],
    'easting':   [320500, 321800, 319200, 325100, 323400, 322000, 321500, 322200],
    'northing':  [742000, 743500, 741000, 740500, 742800, 738500, 736000, 733000],
    'catchment': ['North','North','North','South','South','Main','Main','Main'],
    'richness':  [22, 19, 24, 18, 20, 15, 13, 11],
    'nitrate':   [1.2, 1.8, 0.9, 2.1, 1.5, 2.8, 3.1, 3.4],
}
df_sites = pd.DataFrame(sites_data)
geometry = [Point(e, n) for e, n in zip(df_sites['easting'], df_sites['northing'])]
gdf_sites = gpd.GeoDataFrame(df_sites, geometry=geometry, crs='EPSG:27700')
print(f"GeoDataFrame: {len(gdf_sites)} sites")
print(f"CRS: {gdf_sites.crs}")
print(f"Geometry type: {gdf_sites.geometry.geom_type.unique()}")
print(gdf_sites[['site_id','easting','northing','richness']].to_string(index=False))

---
## Creating and Visualising Geometries

In [ ]:
# Create river reaches (LineStrings)
river_north = LineString([(319200,742000),(320500,742000),(321800,743500),
                           (322000,741500),(322000,738500)])
river_south = LineString([(325100,740500),(323400,742800),(322000,741500)])
river_main  = LineString([(322000,738500),(321500,736000),(322200,733000)])

gdf_rivers = gpd.GeoDataFrame(
    {'river': ['North','South','Main'],
     'length_m': [river_north.length, river_south.length, river_main.length]},
    geometry=[river_north, river_south, river_main],
    crs='EPSG:27700')
print(f"River lengths (m): {gdf_rivers[['river','length_m']].to_string(index=False)}")

# Create catchment polygons (simplified)
poly_north = Polygon([(318000,741000),(323000,741000),(323000,745000),
                       (318000,745000),(318000,741000)])
poly_south = Polygon([(322000,739000),(327000,739000),(327000,744000),
                       (322000,744000),(322000,739000)])
poly_main  = Polygon([(319000,732000),(326000,732000),(326000,741000),
                       (319000,741000),(319000,732000)])
gdf_catch  = gpd.GeoDataFrame(
    {'catchment': ['North','South','Main'],
     'area_m2': [poly_north.area, poly_south.area, poly_main.area]},
    geometry=[poly_north, poly_south, poly_main], crs='EPSG:27700')

fig, ax = plt.subplots(figsize=(9,9))
gdf_catch.plot(ax=ax, column='catchment', cmap='Set2', alpha=0.3, edgecolor='grey', linewidth=1)
gdf_rivers.plot(ax=ax, color='steelblue', linewidth=2)
gdf_sites.plot(ax=ax, column='richness', cmap='YlOrRd', markersize=120,
               edgecolor='black', linewidth=0.5, legend=True,
               legend_kwds={'label':'Species richness','shrink':0.6})
for _, row in gdf_sites.iterrows():
    ax.annotate(row['site_id'], (row.geometry.x+200, row.geometry.y+200), fontsize=7)
ax.set_title('Riparian Monitoring Network
(point=monitoring site, colour=richness)')
ax.set_xlabel('Easting (m)'); ax.set_ylabel('Northing (m)')
plt.tight_layout(); plt.show()

---
## Spatial Operations: Buffer, Intersect, Dissolve

In [ ]:
# Buffer: 500m riparian buffer around river channels
gdf_rivers_buf = gdf_rivers.copy()
gdf_rivers_buf['geometry'] = gdf_rivers_buf.buffer(500)
print("River buffer areas (m²):")
print(gdf_rivers_buf[['river','geometry']].assign(
    area_m2=gdf_rivers_buf.area.round(0)).to_string(index=False))

# Spatial join: which catchment does each site fall in?
gdf_joined = gpd.sjoin(gdf_sites, gdf_catch[['catchment','geometry']],
                        how='left', predicate='within')
print(f"\nSpatial join — sites matched to catchments:")
print(gdf_joined[['site_id','catchment_left','catchment_right']].to_string(index=False))

# Dissolve: merge all catchments into a single study area polygon
gdf_study_area = gdf_catch.dissolve()
print(f"\nDissolved study area: {gdf_study_area.area.values[0]/1e6:.2f} km²")

# Overlay: intersection of north catchment and river buffer
north_catch = gdf_catch[gdf_catch['catchment']=='North']
north_buf   = gdf_rivers_buf[gdf_rivers_buf['river']=='North']
intersection = gpd.overlay(north_catch, north_buf, how='intersection')
print(f"North catchment ∩ river buffer: {intersection.area.values[0]/1e6:.4f} km²")

---
## Distance Calculations and Nearest Neighbours

In [ ]:
# Pairwise distances between monitoring sites (in metres, BNG projected)
from scipy.spatial.distance import cdist
coords = np.array([[geom.x, geom.y] for geom in gdf_sites.geometry])
dist_matrix = cdist(coords, coords)
dist_df = pd.DataFrame(dist_matrix, index=gdf_sites['site_id'],
                         columns=gdf_sites['site_id'])
print("Pairwise distances (m) — first 4 sites:")
print(dist_df.iloc[:4,:4].round(0).to_string())

# Nearest neighbour for each site
nn_distances, nn_indices = [], []
for i in range(len(gdf_sites)):
    dists = dist_matrix[i].copy()
    dists[i] = np.inf
    nn_idx = dists.argmin()
    nn_distances.append(dists[nn_idx])
    nn_indices.append(gdf_sites['site_id'].iloc[nn_idx])
gdf_sites['nn_site']     = nn_indices
gdf_sites['nn_dist_m']   = [round(d,0) for d in nn_distances]
print(f"\nNearest neighbours:")
print(gdf_sites[['site_id','nn_site','nn_dist_m']].to_string(index=False))

In [ ]:
# CRS transformation: BNG -> WGS84 for web mapping
gdf_wgs84 = gdf_sites.to_crs('EPSG:4326')
print("Sites in WGS84 (lon/lat):")
for _, row in gdf_wgs84.iterrows():
    print(f"  {row['site_id']:12s}: lon={row.geometry.x:.5f}, lat={row.geometry.y:.5f}")

# Centroid and bounding box
centroid = gdf_sites.dissolve().centroid.iloc[0]
bbox     = gdf_sites.total_bounds   # [minx, miny, maxx, maxy]
print(f"\nStudy area centroid (BNG): ({centroid.x:.0f}, {centroid.y:.0f})")
print(f"Bounding box: E={bbox[0]:.0f}–{bbox[2]:.0f}, N={bbox[1]:.0f}–{bbox[3]:.0f}")
print(f"Extent: {(bbox[2]-bbox[0])/1000:.1f} km E-W × {(bbox[3]-bbox[1])/1000:.1f} km N-S")
print("\nAlways reproject to metric CRS (e.g. EPSG:27700) before:")
print("  - Computing distances or buffers")
print("  - Calculating polygon areas")
print("  - Running spatial joins requiring accurate predicates")

---

## Common Pitfalls

**1. Computing distances or areas in a geographic CRS (degrees)**  
In WGS84 (EPSG:4326), coordinates are in degrees. A 500m buffer specified in degrees produces wildly incorrect results — 0.0045 degrees is approximately 500m at the equator but varies with latitude. Always reproject to a metric projected CRS (e.g. UTM, British National Grid) before any distance, area, or buffer calculation.

**2. Forgetting to set the CRS when creating a GeoDataFrame from scratch**  
A GeoDataFrame created with `gpd.GeoDataFrame(..., geometry=geoms)` has `crs=None`. Attempting to reproject or spatially join with another GeoDataFrame raises a CRS mismatch error. Always specify `crs=` at construction time or immediately after with `gdf.set_crs()`.

**3. Using `sjoin` with the wrong predicate**  
`predicate="within"` tests if each left geometry is entirely within a right polygon; `predicate="intersects"` matches any overlap. Using `"within"` for points exactly on polygon boundaries may miss matches (depending on topology). For point-in-polygon joins, `"within"` or `"intersects"` usually agree, but verify edge cases.

**4. Dissolving without specifying a column — silently drops all non-geometry columns**  
`gdf.dissolve()` with no `by=` argument merges all rows into one geometry, dropping all attribute columns. Always specify `by="column_name"` to dissolve by group, or explicitly select which columns to retain using `aggfunc=`.

**5. Mixing GeoDataFrames with different CRSs in a spatial operation**  
GeoPandas does not automatically reproject before spatial joins or overlays. Overlaying two GeoDataFrames in different CRSs produces geometrically incorrect results without raising an error. Always verify `gdf1.crs == gdf2.crs` before any spatial operation and reproject one to match the other.
---
*python_methods_library - Samantha McGarrigle*